In [1]:
import numpy as np
import os
from cloudvolume import CloudVolume, Skeleton
import logging
import glob
import json
import neuroglancer
import requests

In [2]:
def make_neuroglancer_url_vneurodata(state,
                                     base_url="http://bigkahuna.corp.alleninstitute.org/neuroglancer",
                                     state_url="https://json.neurodata.io/v1"):
    r = requests.post(state_url, json=state)
    json_url = r.json()["uri"]
    link = f"{base_url}/#!{json_url}"
    print(link)   

def generate_ngl_segmentation_empty(im_data_shape, source_path, out_path, chunk_size, resolution):
    """Create a neuroglancer precomputed segmentation volume from a tiff stack.
       source_path: directory underwhich `swc_files_nm` will be found
       out_path: directory where new the new cloud volume segmentation layer should be generated
       chunk_size: list or array with chunk size for segmentation layer
       resolution: voxel resolution, in nm (passed on to neuroglancer via 'info' file)
    """
 
    info = CloudVolume.create_new_info(
        num_channels    = 1,
        layer_type      = 'segmentation',
        data_type       = 'uint64', # Channel images might be 'uint8'
        # raw, png, jpeg, compressed_segmentation, fpzip, kempressed, compresso
        encoding        = 'compressed_segmentation', 
        resolution      = resolution, # Voxel scaling, units are in nanometers
        voxel_offset    = [0, 0, 0], # x,y,z offset in voxels from the origin
        chunk_size      = chunk_size, # units are voxels
        volume_size     = im_data_shape, # e.g. a cubic millimeter dataset
        skeletons       = 'skeletons'
        )

    vol = CloudVolume(f'file://{out_path}', info=info, compress='', cache=False)
    logging.info(f"Creating cloud volume: {vol.info}")
    vol.commit_info()
    vol.commit_provenance()
        
        
def generate_ngl_skeletons_edit(source_path, out_path):
    """Generate skeletons from SWC files.
       This currently assumes the neuroglancer precomputed volume has already been generated by generate_ngl_segmentation.
       source_path: directory underwhich `swc_files_nm` will be found.
       out_path: directory with the previously generated segmentation layer.
    """

    vol = CloudVolume(f'file://{out_path}', compress='')
    vol.skeleton.meta.info.pop("vertex_attributes", None)
    vol.skeleton.meta.commit_info()

    files = glob.glob(f"{source_path}/*.swc")
    files = sorted(files)
    skel_dir = os.path.join(out_path, "skeletons")
    if not os.path.exists(skel_dir):
        os.makedirs(skel_dir)
    
    segprops = {"@type": "neuroglancer_segment_properties",
            "inline" : {
                "ids" : [],
                "properties" : [
                    {"id": "tags",
                        "type": "tags",
                        "tags" : ["all"],
                        "values" : []
                    },
                    {"id": "length",
                        "type": "number",
                        "data_type" : "float32",
                        "values" : []
                    }
                ]},
            }
    
    sid = 1
    for filename in files:
        # ..../NNNN.swc -> NNNN
        with open(filename, mode='r') as f:
            swc = f.read()
        skel = Skeleton.from_swc(swc)
        skel.id = sid 
 
        vol.skeleton.upload(skel)

        segprops["inline"]["ids"].append(str(sid))
        segprops["inline"]["properties"][0]["values"].append([0])  # tags
        segprops["inline"]["properties"][1]["values"].append(str(skel.cable_length()))
        
        sid += 1
            
    # Write the segment properties
    segment_info_dir = os.path.join(out_path, "skeletons/segment_properties")
    os.makedirs(segment_info_dir, exist_ok=True)
    with open(os.path.join(segment_info_dir, "info"), "w") as f:
        json.dump(segprops, f)
        
    # Re-write info file with added segment_properties
    with open(f'{os.path.join(out_path, "skeletons")}/info', 'r') as f:
        infofile = json.load(f)
    infofile['segment_properties'] = 'segment_properties'
    
    with open(f'{out_path}skeletons/info', 'w') as f:
        json.dump(infofile, f)

In [3]:
s_range = [52,54]
indir = '/ACdata/Users/connorl/skeletons/For_Kevin/S32_Pos52,53,54_MIP0/' ###skeletons
outdir = '/ACdata/Users/connorl/Neuroglancer/S32_Skeletons_Mip0/'
zarr_dir = '/ACdata/Users/kevin/ispim_ome_zarr/H17_x55_S32_230412_highres/H17_x55_S32_230412_highres.zarr/'
mip = 0
skel_trans = [23000,0,0]

In [ ]:
###Create precomputed volume for skeletons
for n in range(s_range[0],s_range[1]+1):
    pos_dir = indir + 'Pos' + str(n) + "/"
    os.makedirs(outdir + 'Pos' + str(n) + "/", exist_ok=True)
    generate_ngl_segmentation_empty([288,288,21586], pos_dir + "Reconnected/reconnected_skeletons/" , outdir + 'Pos' + str(n) + "/", [512, 512, 64], [406, 406, 1997.72])
    generate_ngl_skeletons_edit(pos_dir + "Reconnected/reconnected_skeletons/" , outdir + 'Pos' + str(n) + "/")

In [4]:
###Load zarr and skeletons into neuroglancer instance
ip = 'localhost' 
port = 9999
neuroglancer.set_server_bind_address(bind_address=ip,bind_port=port)
viewer=neuroglancer.Viewer()

#load image and skeletons
for n in range(s_range[0],s_range[1]+1):
    
    ###Extract translation from zarr metadata
    zarr_attr = zarr_dir + 'highres_Pos' + str(n) + '/.zattrs'
    f = open(zarr_attr)
    data = json.load(f)
    
    #pull translations and scales for image and skeletons
    trans_im = np.array(data['multiscales'][0]['coordinateTransformations'][0]['translation'])[2:5]
    scale_im = np.array(data['multiscales'][0]['datasets'][0]['coordinateTransformations'][0]['scale'][2:5])
    trans_seg = np.array([trans_im[0]/scale_im[0],trans_im[1]/scale_im[1],trans_im[2]/scale_im[2]]).astype(float)
    scale_seg = np.array(data['multiscales'][0]['datasets'][mip]['coordinateTransformations'][0]['scale'][2:5])

    #alter dimension order and set dimension scale
    dim_im = neuroglancer.CoordinateSpace(
                names=['z', 'y', 'x', 't'],
                units='nm',
                scales=scale_im*1000,
            )

    dim_seg = neuroglancer.CoordinateSpace(
                names=['z', 'y', 'x', 't'],
                units='nm',
                scales=scale_seg*1000,
            )

    #create coordinate transforms to adjust for zarr mip level and translations
    tr_im = neuroglancer.CoordinateSpaceTransform(input_dimensions = dim_im, output_dimensions =  dim_im) 
    tr_seg = neuroglancer.CoordinateSpaceTransform(input_dimensions = dim_seg, output_dimensions =  dim_seg,
                                                  matrix = np.array([[1,0,0,trans_seg[0]/(1+mip) + skel_trans[0]],[0,1,0,trans_seg[1]/(1+mip) + skel_trans[1]],[0,0,1,trans_seg[2]/(1+mip) + skel_trans[2]]]))

    
    ###load image and skeletons
    with viewer.txn() as s:
        s.dimensions = dim_im
        s.layers['Image_Pos' + str(n)] = neuroglancer.ImageLayer(source=['zarr://http://bigkahuna.corp.alleninstitute.org' + zarr_dir + 'highres_Pos'+str(n)])
        s.layers['Image_Pos' + str(n)].layer.source[0].transform  = tr_im
        
    with viewer.txn() as s:
        s.layers['Skel_Pos' + str(n)] = neuroglancer.SegmentationLayer(source=['precomputed://http://bigkahuna.corp.alleninstitute.org' + outdir + 'Pos'+str(n)+'/skeletons/'])
        s.layers['Skel_Pos' + str(n)].layer.source[0].transform  = tr_seg
    
print(viewer)

http://localhost:9999/v/9df39e7b4fc9158718a7232e73236c9bd8cb1015/


In [ ]:
###Create shareable neuroglancer link
view = s.to_json()
make_neuroglancer_url_vneurodata(view)